# Signing and verifying a model with Cosign (and a look at OpenSSF model signing)

> **Not executed by the repo author: run in Colab or in your course lab; versions may need updates.** Check each tool's current documentation and `--help` before trusting a flag or an API name.

**Goal.** Sign a model file, verify it, tamper with it, and see verification fail.

| | |
|---|---|
| Runtime | CPU, Linux (Colab is fine). Local Windows/macOS also works with the matching Cosign binary. |
| Time | 20 minutes |
| Offline twin | `labs/model-signing/` (HMAC manifest signing, with tests; do it first) |

**Keyless vs key-based.** Cosign can sign **keyless** (you log in with an identity provider such as GitHub or Google, you get a
short-lived certificate, and the signature is recorded in a public transparency log) or with a **key pair** you manage.
Keyless is the recommended path for CI. It needs an interactive login in a browser, which is awkward in Colab, so this
notebook uses a **key pair** for the hands-on part and only shows the keyless commands.

**Privacy note.** Keyless signing writes your identity (for example an email address) and a signature record into a public
log. Use a throw-away identity for practice.

## 1. Install Cosign

In [ ]:
# Cosign release v2.4.1 for Linux amd64. Check https://github.com/sigstore/cosign/releases for the current version
# and VERIFY THE CHECKSUM listed on the release page before you run the binary.
!curl -sSL -o cosign https://github.com/sigstore/cosign/releases/download/v2.4.1/cosign-linux-amd64
!sha256sum cosign
!chmod +x cosign && ./cosign version

Compare the printed `sha256sum` with the value on the release page. Stop if it differs.

## 2. Make a stand-in model file

In [ ]:
!pip install -q safetensors==0.4.5 numpy
import numpy as np
from safetensors.numpy import save_file
save_file({"w": np.arange(1000, dtype="float32")}, "model.safetensors")
!ls -l model.safetensors && sha256sum model.safetensors

## 3. Sign with a key pair

`cosign generate-key-pair` writes `cosign.key` (private) and `cosign.pub` (public). It asks for a password; here we pass one
through an environment variable for the demo only. **Never commit `cosign.key`.** In real life use a KMS, an HSM or keyless signing.

Flags differ between Cosign versions (for example, whether a transparency-log upload is on by default). Run
`./cosign sign-blob --help` and `./cosign verify-blob --help` and adjust.

In [ ]:
import os
os.environ["COSIGN_PASSWORD"] = "demo-only-password"
!./cosign generate-key-pair
!./cosign sign-blob --key cosign.key --tlog-upload=false --output-signature model.sig model.safetensors --yes
!ls -l model.sig

In [ ]:
!./cosign verify-blob --key cosign.pub --signature model.sig --insecure-ignore-tlog=true model.safetensors

Expected: a short "Verified OK" message. If your version rejects a flag, read its `--help`.

## 4. Tamper and verify again

In [ ]:
data = bytearray(open("model.safetensors", "rb").read())
data[100] ^= 1                                # flip one bit
open("model.safetensors", "wb").write(bytes(data))
!./cosign verify-blob --key cosign.pub --signature model.sig --insecure-ignore-tlog=true model.safetensors || echo "verification failed, as it should"

## 5. Keyless signing (read only; needs a browser login)

In CI (for example GitHub Actions with OIDC) the flow is roughly:

```bash
cosign sign-blob --bundle model.sigstore.json model.safetensors
cosign verify-blob --bundle model.sigstore.json \
  --certificate-identity "https://github.com/ORG/REPO/.github/workflows/release.yml@refs/tags/v1.0.0" \
  --certificate-oidc-issuer "https://token.actions.githubusercontent.com" \
  model.safetensors
```

The verifier states **who** is allowed to have signed (the identity) and **which identity provider** vouches for them.
The identity and issuer strings above are placeholders. Without those two checks, "signed" proves little.

## 6. OpenSSF Model Signing (OMS)

The OpenSSF *model-signing* project signs a whole **model directory** (many files, large files, hashing in chunks) and
writes a Sigstore bundle. Its v1.0 was announced in April 2025. Its Python package and command line may differ by version.
Treat the commands below as a sketch and read `--help`.

In [ ]:
!pip install -q model-signing==1.0.0
!model_signing --help

In [ ]:
# Typical shape (check --help for the real subcommands and flags in your version):
#   model_signing sign <model_dir_or_file>   -> writes a signature file (Sigstore bundle format)
#   model_signing verify <model_dir_or_file> --signature <file> --identity <signer> --identity_provider <issuer>
# Keyless commands open a browser login, so run them on your own machine, not in a shared notebook.

## Questions

1. What does the signature prove, and what does it **not** prove? (It proves who signed and that the bytes did not change. It does not prove the model is safe or unbiased.)
2. Why does the verifier need to know the expected signer identity?
3. Where would you store the verification step in your pipeline? (At download time, before load, and again in the deployment admission check.)